# Stage Three - The Cross-Encoder Reranker

**Upside, not a requirement.** This comes after stages one and two, or not at
all. The reason to read it now is that deciding *whether* to build it is itself
a good finding, and you already have the number that decides it.

---

## The two machines

**Bi-encoder** (what stages one and two built). Converts each document into 384
numbers ahead of time - a point in space. At query time it converts the query to
a point and finds the nearest document points. Milliseconds.

It is fast *precisely because* it summarised each document blind, before knowing
what would be asked of it. That blindness is also its ceiling.

**Cross-encoder** (this stage). Reads the query and one document **together** and
scores relevance directly. Far more accurate, because it can attend to exactly
which part of the document answers *this* query. But nothing can be precomputed:
scoring 5,000 documents means 5,000 forward passes. Minutes.

So a cross-encoder cannot be a search engine. It can only be a second pass over a
shortlist someone else produced.

## Similarity is not relevance

This is the deeper reason the second pass helps. A bi-encoder ranks by embedding
similarity, which is a *proxy* for relevance. Usually a good one. But an abstract
can be enormously similar to a query - same topic, same vocabulary - while
answering nothing.

Fine-tuning makes the proxy better calibrated to your domain. It is still a proxy,
computed from a 384-number summary made before anyone knew what would be asked.

A cross-encoder breaks that ceiling because it asks a different question: not
*"are these similar"* but *"does this document answer this query"*.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

here = Path.cwd()
while not (here / "configs").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
sys.path.insert(0, str(here / "src"))

import json
import time

import pandas as pd
import torch

from config import load_config
from ingest import doc_text, load_frozen_split, load_raw_corpus
from runfile import read_run

cfg = load_config("configs/scifact.yaml")
queries, qrels, manifest = load_frozen_split(cfg.dataset, "test")
corpus = load_raw_corpus(cfg.dataset)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print(f"{len(queries)} test queries, {len(corpus)} documents")

## 1. Should you build it at all?

This is the first question, not the last, and one number answers it.

| | What happened | Can reranking fix it? |
|---|---|---|
| **Failure mode A** | The right document sits at rank 800. Never made the shortlist. | **No** |
| **Failure mode B** | The right document is at rank 7. In the pile, wrong position. | **Yes** |

**Recall@100 tells you which one you are in.** Of the genuinely relevant
documents, how many made the top 100?

- Around **0.94** - the answers are in the pile and just need reordering. A
  reranker has real room to work.
- Around **0.55** - half the answers never made the shortlist. No amount of
  reranking recovers them. The problem is upstream, in the retriever or the
  chunking, and building a reranker would be wasted effort.

You measured this for free in stage one.

In [ ]:
df = pd.read_csv("results/metrics.csv")
df[["model_name", "index_type", "ndcg@10", "recall@50", "recall@100"]]

### Reading your own numbers

The dense baseline gives **recall@100 = 0.9450** against **nDCG@10 = 0.7127**.

The gap of 0.232 is the space a reranker can work in. The right answer is in the
top 100 for 95% of queries, but well-ranked for only 71%. That is squarely
failure mode B, and it is the situation reranking exists for.

**Now look at recall@50 versus recall@100: 0.9317 against 0.9450.**

Going from a shortlist of 50 to one of 100 buys you 1.3 points of ceiling.
Cross-encoder cost is *linear* in shortlist depth, so the 100-deep shortlist costs
exactly double for that 1.3 points. Checking this before choosing a depth is the
kind of thing that separates an engineering decision from a default.

In [ ]:
r50, r100 = 0.9317, 0.9450
print(f"recall@50  {r50:.4f}   <- ceiling if you rerank 50")
print(f"recall@100 {r100:.4f}   <- ceiling if you rerank 100")
print(f"difference {r100 - r50:.4f}  for 2x the cross-encoder compute")

## 2. What reranking actually does, on one query

Take a query where the bi-encoder put the right answer outside the top 10 - the
kind of query nDCG@10 scores zero. Then let the cross-encoder reorder the
shortlist and watch where it lands.

In [ ]:
base_run = read_run("data/runs/scifact_base_test.trec")

# Find a query where the relevant document is in the top 100 but NOT the top 10.
# Those are exactly the queries a reranker exists to rescue.
candidates = []
for qid, judged in qrels.items():
    ranked = sorted(base_run[qid].items(), key=lambda kv: -kv[1])
    ids = [d for d, _ in ranked[:100]]
    for did in judged:
        if did in ids and ids.index(did) >= 10:
            candidates.append((qid, did, ids.index(did) + 1))

print(f"{len(candidates)} query/document pairs are in the top 100 but not the top 10")
demo_qid, demo_did, demo_rank = candidates[0]
print(f"\nusing query {demo_qid}, relevant doc at rank {demo_rank}")
print("query:", queries[demo_qid][:100])

In [ ]:
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2",
                        max_length=512, device=device)

ranked = sorted(base_run[demo_qid].items(), key=lambda kv: -kv[1])[:50]
shortlist = [d for d, _ in ranked]

# NOTE: raw query text, no BGE prefix. Explained in the next section.
pairs = [[queries[demo_qid], doc_text(corpus[d])] for d in shortlist]
scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

new_order = [d for d, _ in sorted(zip(shortlist, scores), key=lambda kv: -kv[1])]

print(f"relevant document {demo_did}")
print(f"  bi-encoder rank : {shortlist.index(demo_did) + 1}")
print(f"  reranked rank   : {new_order.index(demo_did) + 1}")

Same 50 documents in both cases. Nothing was retrieved, nothing was dropped. Only
the order changed, and that is the entire mechanism.

### The invariant that proves it is working

**Recall@100 must be identical before and after reranking the top 100.** Not
similar - identical. It is the same set of documents.

If your recall moves, the reranker has a bug: it dropped documents, or it pushed
some out of the top 100 by mishandling the part of the list it did not rescore.
`rerank.py` prints this reminder every time it runs.

In [ ]:
print("before:", sorted(shortlist)[:5], "...")
print("after :", sorted(new_order)[:5], "...")
print("same set of documents:", set(shortlist) == set(new_order))

## 3. Two design choices in `rerank.py`

### No query prefix, and this is not an oversight

The BGE instruction - *"Represent this sentence for searching relevant
passages: "* - is an instruction to a **bi-encoder** about how to build a
standalone embedding.

A cross-encoder reads the query and document as one joined sequence, and this one
was trained on raw MS MARCO pairs. Prepending BGE's instruction feeds it text it
never saw during training. Different model, different contract.

So `rerank.py` uses raw query text and records `query_prefix_used=no` in
`metrics.csv`. The prefix column exists precisely so this stays recoverable
later, rather than being a thing you have to remember.

### The untouched tail is kept, not discarded

Documents past rank k are not rescored. But throwing them away would destroy
recall@100 whenever k is 50, and make the run incomparable to the one it came
from.

So they are retained and offset so that **every reranked document outranks every
untouched one**, with their relative order preserved. Recall stays identical by
construction, which is what lets you attribute any nDCG change entirely to
reordering.

In [ ]:
print('''
if tail:
    floor = min(rescored.values())        # worst reranked score
    span = max(abs(floor), 1.0)
    for rank, (did, old) in enumerate(tail, start=1):
        rescored[did] = floor - span - rank * 1e-6
''')
print("Cross-encoder scores are unbounded logits, so the offset is computed")
print("from the actual floor rather than assuming any particular range.")

## 4. The cost, which you must report

A cross-encoder is orders of magnitude slower per query than the search it sits
on top of. A gain that costs 300ms per query is a completely different
proposition from one that costs 5ms, and reporting the gain without the cost is
not an honest result.

In [ ]:
t0 = time.perf_counter()
_ = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
elapsed = time.perf_counter() - t0

print(f"device                {device}")
print(f"documents rescored    {len(pairs)}")
print(f"time for one query    {1000 * elapsed:.0f} ms")
print(f"pairs per second      {len(pairs) / elapsed:.0f}")
print()
print(f"extrapolated to 300 queries at depth 50 : {300 * elapsed:.0f} s")
print(f"extrapolated to 300 queries at depth 100: {600 * elapsed:.0f} s")
print()
print("Compare that to the bi-encoder search it follows, which answered the")
print("same 300 queries in well under a second against a flat index.")

### Report these separately, per step 7 of the README

- index build time and peak memory
- on-disk index size
- query encoding latency, p50 and p95, at batch size 1
- search latency per index configuration
- rerank latency at each depth
- end to end

Warm up with 50 discarded queries, then measure over at least 500. Pin threads
with `faiss.omp_set_num_threads(1)` for a reproducible single-query number, and
report throughput separately with threads unpinned.

**Record the hardware.** An unlabelled latency number is worthless, and Colab
assigns different CPUs between sessions.

## 5. Running the whole stage

`rerank.py` reads a run file and writes a run file. It never imports
`build_index`, `retrieve`, or `train`, and nothing in stage two changes to
accommodate it. **`evaluate.py` cannot tell a cross-encoder was involved.**

That is the payoff of the interface decision made in stage one.

In [ ]:
print('''
# one depth
python src/rerank.py --config configs/scifact.yaml \\
    --run data/runs/scifact_base_test.trec --top-k 50

# both depths, plus evaluation of each
bash scripts/run_stage3.sh configs/scifact.yaml data/runs/scifact_base_test.trec
''')

### Which run should you rerank?

The best one you have. Reranking your *fine-tuned* stage two run is the
interesting experiment, because the two techniques attack different failure
modes and their gains should partly add:

- fine-tuning moves documents **into** the shortlist (mode A)
- reranking reorders **within** the shortlist (mode B)

Reranking the base run as well gives you a clean decomposition of where each
point came from.

## 6. Distillation - the upgrade beyond off-the-shelf

Everything above uses `ms-marco-MiniLM-L-6-v2` straight off the shelf. It is
trained on MS MARCO, which is web search, not your domain.

The stage-three-proper version trains your own cross-encoder by **distillation**:

1. Take a large, slow, accurate cross-encoder as the teacher.
2. Have it score your (query, document) pairs, including the mined negatives you
   already produced in stage two.
3. Train a small student cross-encoder to reproduce those scores.

You end up with a reranker adapted to your domain that runs at the small model's
speed. It reuses `data/hard_negatives/` directly, which is why stage two produced
that file in a reusable form.

**Do this only after stages one and two are finished and written up.** The
off-the-shelf reranker already tells you whether reranking helps at all on your
data, which is the finding that matters. Distillation improves a result you
have already established.

## 7. What to write down

Report, for each depth you tried:

| Column | Why |
|---|---|
| nDCG@10 before and after | where a reranker shows up |
| MRR@10 before and after | reranking mostly moves the *first* relevant hit |
| recall@100 before and after | must be identical - it is your correctness check |
| ms per query | the cost that makes the gain interpretable |

And state the honest framing: reranking cannot fix retrieval. It reorders what
retrieval already found. If your recall@100 had been 0.55, the correct finding
would have been *"a reranker is not worth building here"*, and that is a real
result too.